In [ ]:
import pandas as pd

path = "../data/"

races = pd.read_csv(path + "races.csv", na_values=["\\N"])
results = pd.read_csv(path + "results.csv", na_values=["\\N"])
circuits = pd.read_csv(path + "circuits.csv", na_values=["\\N"])
quali = pd.read_csv(path + "qualifying.csv", na_values=["\\N"])
status = pd.read_csv(path + "status.csv", na_values=["\\N"])

In [ ]:
# Drop unneeded car number 
#results = results.drop(columns=["number"])
# Sort results table starting from 2024 and descending, and then by position order ascending
results = results.sort_values(['raceId', 'positionOrder'], ascending=[False, True])

In [58]:
# Sense-check race ID year value
races.loc[races['raceId'] == 1144, ['raceId', 'year']]

,raceId,year
1124,1144,2024


DATA EXPLORATION AND CLEANING

In [ ]:
# Get number of rows and columns in each table
print("races: ", races.shape, 
      "\nresults: ", results.shape,
      "\ncircuits: ", circuits.shape,
      "\nqualifying: ", quali.shape)

races:  (1125, 18) 
results:  (26759, 18) 
drivers:  (861, 9) 
constructors:  (212, 5) 
circuits:  (77, 9) 
qualifying:  (10494, 9)


In [ ]:
print("races columns: ", races.columns.tolist(),
      "\nresults columns: ", results.columns.tolist(),
      "\nquali columns: ", quali.columns.tolist())

races columns:  ['raceId', 'year', 'round', 'circuitId', 'name', 'date', 'time', 'url', 'fp1_date', 'fp1_time', 'fp2_date', 'fp2_time', 'fp3_date', 'fp3_time', 'quali_date', 'quali_time', 'sprint_date', 'sprint_time'] 
results columns:  ['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid', 'position', 'positionText', 'positionOrder', 'points', 'laps', 'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'statusId'] 
quali columns:  ['qualifyId', 'raceId', 'driverId', 'constructorId', 'number', 'position', 'q1', 'q2', 'q3']


In [86]:
# Filter races for 2014-2024 seasons (post introduction of mandatory Hybrid engine)
hybridEra = races[(races["year"] >= 2014) & (races["year"] <= 2024)]
# Filter results based on races present in filtered hybridEra races table
hybridResults = results[results["raceId"].isin(hybridEra["raceId"])]
#hybridResults
hybridQuali = quali[quali["raceId"].isin(hybridEra["raceId"])]
hybridQuali

,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3
5857,5860,900,1,131,44,1,1:31.699,1:42.890,1:44.231
5858,5861,900,817,9,3,2,1:30.775,1:42.295,1:44.548
5859,5862,900,3,131,6,3,1:32.564,1:42.264,1:44.595
5860,5863,900,825,1,20,4,1:30.949,1:43.247,1:45.745
5861,5864,900,4,6,14,5,1:31.388,1:42.805,1:45.819
...,...,...,...,...,...,...,...,...,...
10489,10547,1144,848,3,23,16,1:23.821,NaN,NaN
10490,10548,1144,855,15,24,17,1:23.880,NaN,NaN
10491,10549,1144,1,131,44,18,1:23.887,NaN,NaN
10492,10550,1144,861,3,43,19,1:23.912,NaN,NaN


In [ ]:
print("min year: ", hybridEra["year"].min(),
      "\nmax year: ", hybridEra["year"].max(),
      "\nraces per year: ", hybridEra["year"].value_counts().sort_index())


min year:  2014 
max year:  2024 
races per year:  year
2014    19
2015    19
2016    21
2017    20
2018    21
2019    21
2020    17
2021    22
2022    22
2023    22
2024    24
Name: count, dtype: int64


In [87]:
print("races NaN: ", hybridEra.isnull().sum())
print("results NaN: ", hybridResults.isnull().sum())
print("qualifying NaN: ", hybridQuali.isnull().sum())

races NaN:  raceId           0
year             0
round            0
circuitId        0
name             0
date             0
time             0
url              0
fp1_date       138
fp1_time       160
fp2_date       138
fp2_time       160
fp3_date       156
fp3_time       175
quali_date     138
quali_time     160
sprint_date    210
sprint_time    213
dtype: int64
results NaN:  resultId              0
raceId                0
driverId              0
constructorId         0
number                0
grid                  0
position            721
positionText          0
positionOrder         0
points                0
laps                  0
time               2219
milliseconds       2219
fastestLap          217
rank                  0
fastestLapTime      217
fastestLapSpeed     217
statusId              0
dtype: int64
qualifying NaN:  qualifyId           0
raceId              0
driverId            0
constructorId       0
number              0
position            0
q1                 51
q2 

In [114]:
missingPos = hybridResults[hybridResults["position"].isnull()]

print(len(missingPos))
print(missingPos["positionText"].value_counts(), missingPos["positionOrder"].describe())

721
positionText
R    680
W     28
D     11
N      1
E      1
Name: count, dtype: int64 count    721.000000
mean      18.690707
std        1.758325
min       12.000000
25%       18.000000
50%       19.000000
75%       20.000000
max       22.000000
Name: positionOrder, dtype: float64


In [94]:
print(hybridResults["grid"].value_counts().sort_index())

grid
0      79
1     228
2     226
3     228
4     228
5     228
6     228
7     227
8     227
9     228
10    226
11    226
12    228
13    226
14    227
15    226
16    226
17    230
18    222
19    216
20    176
21     36
22     34
Name: count, dtype: int64


In [ ]:
print(hybridResults[hybridResults["grid"] == 0]["positionText"].value_counts(),
      "\n", hybridResults[hybridResults["grid"] == 0]["positionText"].isnull().value_counts())

positionText
R     13
17     9
10     7
13     7
W      6
16     6
18     6
15     6
11     4
5      3
12     3
14     3
9      2
20     1
19     1
8      1
7      1
Name: count, dtype: int64 
 position
False    60
True     19
Name: count, dtype: int64
